# m33 lab · From map to digital twin

*Open EP Mapping Lab. Education only: not for diagnosis, treatment or any clinical decision.*

You build a small twin from a mapping case and test it honestly. The case is synthetic, so the truth is known
and every claim the twin makes can be checked. The lab stops at writing files that openCARP can read; it never
runs openCARP, which is not part of this course (see the last section to run it yourself).

| Part | What you do | Data |
| --- | --- | --- |
| A | Build a mapping case: an atrium-sized sphere with a pacing site, a slow region and a scar; acquire points with position and timing errors | synthetic |
| B | Choose the twin's mesh: eikonal error and run time on three resolutions | synthetic |
| C | Label regions from voltage, and see how the threshold moves the labels | synthetic |
| D | Calibrate regional speeds, with the timing offset solved in closed form | synthetic |
| E | Cross-validate competing twins, and refit an over-split twin from several starts | synthetic |
| F | Export mesh, region tags and fibres in openCARP's formats; turn speeds into conductivity scale factors | synthetic |

Each part ends with **checks**: lines that pass with the code as given. If you change something and a check fails,
that is information, not an error to hide. The whole lab runs in about a minute.

In [ ]:
import sys
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt

# Find the shared model code in labs/python/epml. In Colab, fetch the course repository first.
try:
    import epml
except ImportError:
    if "google.colab" in sys.modules:
        import subprocess
        subprocess.run(["git", "clone", "--depth", "1",
                        "https://github.com/raghavendrasri/open-ep-mapping-lab",
                        "open-ep-mapping-lab"], check=True)
        sys.path.insert(0, "open-ep-mapping-lab/labs/python")
    else:
        sys.path.insert(0, str(Path.cwd() / "labs" / "python"))
    import epml

plt.rcParams.update({"figure.figsize": (8, 3.2), "axes.grid": True, "axes.spines.top": False, "axes.spines.right": False})

import json
import tempfile
import time

from epml import mapping as mp
from epml import twin

## Part A · The case

The "atrium" is a sphere of radius 20 mm: a size chosen for illustration, not a measurement. It is paced at one
point. Normal tissue conducts at 1.0 m/s, and a slow region at 0.6 m/s; these are the values reported for
normal-voltage and low-voltage left atrium in persistent AF (Heil et al. 2023). A scar does not conduct. The
sizes and places of the slow region and the scar are illustrative.

The ground truth is the eikonal model of m30 on a fine mesh (about 1.5-mm edges). Note: 1 m/s = 1 mm/ms.

In [ ]:
R = 20.0                                   # mm, illustrative atrium-sized sphere
V_FAST, V_SLOW = 1.0, 0.6                  # m/s = mm/ms (Heil et al. 2023)
SLOW_DIR, SLOW_R = [0.2, 0.9, 0.0], 12.0   # direction of the slow region's centre, radius in mm (illustrative)
SCAR_DIR, SCAR_R = [0.5, -0.5, 0.7], 8.0   # same for the scar (illustrative)


def unit(v):
    v = np.asarray(v, float)
    return v / np.linalg.norm(v)


def cap(points, direction, radius_mm):
    """True for points within `radius_mm` (measured along the sphere) of a direction."""
    cosang = points @ unit(direction) / np.linalg.norm(points, axis=1)
    return np.arccos(np.clip(cosang, -1, 1)) * R <= radius_mm


V4, F4 = mp.icosphere(R, 4)                # fine mesh: the truth
c4 = V4[F4].mean(axis=1)
speed4 = np.where(cap(c4, SCAR_DIR, SCAR_R), 0.0, np.where(cap(c4, SLOW_DIR, SLOW_R), V_SLOW, V_FAST))
pace4 = int(np.argmin(np.linalg.norm(V4 - [-R, 0, 0], axis=1)))
T_true = mp.eikonal(V4, F4, pace4, speed4)
print(f"truth mesh: {len(V4)} vertices; activation spans {np.nanmax(T_true[np.isfinite(T_true)]):.1f} ms")

Now acquire points the way a mapping system would, with three kinds of error. Positions carry a random error
(1 mm in each direction here) and are projected back onto the surface (eq-m30-projection). Activation times carry
annotation noise (1 ms) and an unknown reference offset (25 ms). Points inside the scar, away from its edge, are
never activated. Each point also has a voltage. Because bipolar
voltage depends on electrodes and wavefront direction (m03), it is kept in **relative units** here: about 1 in
normal tissue, 0.35 in the slow region and 0.08 in scar, with random spread. Scar points have a voltage but no
activation time. All these error sizes and voltages are illustrative inputs; typical clinical values are still to
be sourced.

In [ ]:
rng = np.random.default_rng(33)
N = 200
idx = rng.choice(len(V4), N, replace=False)
measured = V4[idx] + rng.normal(0, 1.0, (N, 3))          # 1 mm position error (illustrative)
points, _ = mp.project_to_mesh(measured, V4, F4)
OFFSET = 25.0                                             # ms, unknown to the twin
lat = T_true[idx] + OFFSET + rng.normal(0, 1.0, N)        # 1 ms annotation noise (illustrative)
in_scar = cap(V4[idx], SCAR_DIR, SCAR_R)
in_slow = cap(V4[idx], SLOW_DIR, SLOW_R)
voltage = np.where(in_scar, 0.08, np.where(in_slow, 0.35, 1.0)) * rng.lognormal(0, 0.25, N)
has_lat = np.isfinite(lat)
print(f"{N} points: {in_scar.sum()} in the scar, {(~has_lat).sum()} of them with no activation time; "
      f"{in_slow.sum()} in the slow region")

lon = lambda p: np.degrees(np.arctan2(p[:, 1], p[:, 0]))
lat_deg = lambda p: np.degrees(np.arcsin(np.clip(p[:, 2] / np.linalg.norm(p, axis=1), -1, 1)))
fig, ax = plt.subplots(1, 2, figsize=(9, 3.2))
s = ax[0].scatter(lon(points[has_lat]), lat_deg(points[has_lat]), c=lat[has_lat], s=14)
fig.colorbar(s, ax=ax[0], label="activation time (ms)")
s = ax[1].scatter(lon(points), lat_deg(points), c=voltage, s=14, cmap="viridis", vmin=0, vmax=1.3)
fig.colorbar(s, ax=ax[1], label="relative voltage")
for a in ax:
    a.set_xlabel("longitude (deg)"); a.set_ylabel("latitude (deg)")
ax[0].set_title("Activation times"); ax[1].set_title("Voltages")
plt.tight_layout(); plt.show()

In [ ]:
# Checks
assert np.all(np.abs(np.linalg.norm(points, axis=1) - R) < 0.05)   # projected points lie on the surface
assert np.all(in_scar[~has_lat]) and (~has_lat).sum() > 0          # only scar points lack an activation time
assert np.nanmax(T_true[np.isfinite(T_true)]) > 2 * R              # the wave goes around the scar

## Part B · The twin's mesh

Fitting runs the forward model hundreds of times, so the twin uses a coarser mesh than the truth. How coarse?
With uniform speed the exact answer is known (great-circle distance / speed), so the error of each resolution can
be measured.

In [ ]:
rows = []
for sub in (2, 3, 4):
    V, F = mp.icosphere(R, sub)
    t = time.perf_counter(); T = mp.eikonal(V, F, 0, 1.0); dt = time.perf_counter() - t
    g = R * np.arccos(np.clip(V @ V[0] / R**2, -1, 1))
    far = g > 5
    edge = np.linalg.norm(V[F[:, 0]] - V[F[:, 1]], axis=1).mean()
    rows.append((sub, len(V), edge, np.mean(np.abs(T[far] - g[far]) / g[far]), dt))
    print(f"subdivision {sub}: {len(V):5d} vertices, edge {edge:.1f} mm, mean error {rows[-1][3]:.1%}, {dt*1000:.0f} ms per solve")

V3, F3 = mp.icosphere(R, 3)                # the twin's mesh
c3 = V3[F3].mean(axis=1)
area3 = 0.5 * np.linalg.norm(np.cross(V3[F3[:, 1]] - V3[F3[:, 0]], V3[F3[:, 2]] - V3[F3[:, 0]]), axis=1)
node = np.argmin(np.linalg.norm(points[:, None, :] - V3[None], axis=2), axis=1)  # nearest twin vertex per point
pace3 = int(np.argmin(np.linalg.norm(V3 - [-R, 0, 0], axis=1)))

In [ ]:
# Checks
assert rows[1][3] < 0.03                       # the twin mesh is within 3% on average
assert rows[1][4] < rows[2][4]                 # and faster than the fine mesh
assert np.allclose(V4[:len(V3)], V3)           # subdivision keeps the coarse vertices, so the meshes nest

## Part C · Regions from voltage

A twin needs to know which tissue is which. Here the only clue is voltage at the points. Interpolate it onto the
twin's mesh (inverse-distance weights over the 8 nearest points, eq-m30-idw with straight-line distance), average
over each triangle, and call a triangle "low voltage" below a threshold. The truth is known, so the labels can be
scored with the Dice overlap: twice the shared area divided by the sum of the two areas.

The thresholds here are on a relative scale and illustrative. Clinical thresholds are conventions tied to specific
electrodes and studies (m03, m23).

In [ ]:
d = np.linalg.norm(V3[:, None, :] - points[None], axis=2)
nearest = np.argsort(d, axis=1)[:, :8]
w = 1 / np.maximum(np.take_along_axis(d, nearest, axis=1), 1e-6) ** 2
v_vertex = (w * voltage[nearest]).sum(axis=1) / w.sum(axis=1)
v_tri = v_vertex[F3].mean(axis=1)
true_low = cap(c3, SLOW_DIR, SLOW_R) | cap(c3, SCAR_DIR, SCAR_R)


def dice(a, b):
    return 2 * area3[a & b].sum() / (area3[a].sum() + area3[b].sum())


thresholds = np.linspace(0.15, 0.9, 16)
scores = [dice(v_tri < th, true_low) for th in thresholds]
fractions = [area3[v_tri < th].sum() / area3.sum() for th in thresholds]
fig, ax = plt.subplots(1, 2, figsize=(9, 3))
ax[0].plot(thresholds, scores, marker="o"); ax[0].set_xlabel("threshold (relative voltage)"); ax[0].set_ylabel("Dice overlap")
ax[1].plot(thresholds, fractions, marker="o", label="labeled low")
ax[1].axhline(area3[true_low].sum() / area3.sum(), color="k", ls="--", label="true low")
ax[1].set_xlabel("threshold (relative voltage)"); ax[1].set_ylabel("share of surface"); ax[1].legend()
plt.tight_layout(); plt.show()
TH = 0.5
print(f"threshold {TH}: labels {area3[v_tri < TH].sum() / area3.sum():.1%} of the surface as low "
      f"(truth {area3[true_low].sum() / area3.sum():.1%}), Dice {dice(v_tri < TH, true_low):.2f}")

Interpolation smooths voltage across the border between low and normal tissue, so a threshold halfway between
the two levels labels less than the true area; a higher threshold recovers more of it. The best threshold depends
on how sparse the points are, so a threshold tuned on one map does not transfer to another.

In [ ]:
# Checks
assert dice(v_tri < TH, true_low) > 0.6                                  # the labels find most of the low region
assert area3[v_tri < TH].sum() < area3[true_low].sum()                   # but a mid threshold under-labels it
assert max(scores) > dice(v_tri < 0.2, true_low) + 0.3                   # a too-strict threshold misses most of it

## Part D · Calibrate

Fit one speed per region so the twin's activation times match the measured ones, with the offset solved in closed
form (eq-m33-offset). Four twins:

- **oracle**: the true regions, with scar held at 0 (block). Not available in real life; a reference.
- **1 region**: one speed everywhere.
- **low < 0.5** and **low < 0.65**: normal and low-voltage regions from Part C, with two different thresholds.

In [ ]:
oracle = np.where(cap(c3, SCAR_DIR, SCAR_R), 2, cap(c3, SLOW_DIR, SLOW_R).astype(int))
twins = {
    "oracle": (oracle, {2: 0.0}),
    "1 region": (np.zeros(len(F3), int), {}),
    "low < 0.5": ((v_tri < 0.5).astype(int), {}),
    "low < 0.65": ((v_tri < 0.65).astype(int), {}),
}


def conducting_vertices(region, fixed):
    """Vertices that belong to at least one triangle the twin lets conduct."""
    live = np.array([fixed.get(r, 1.0) > 0 for r in region])
    return np.unique(F3[live])


# A point whose nearest twin vertex sits inside a twin's block region can never be activated by that
# twin. Use the same points for every twin: those with an activation time that every twin can reach.
reach = set.intersection(*[set(conducting_vertices(*twins[n])) for n in twins])
use = has_lat & np.isin(node, list(reach))
pts, t_meas = node[use], lat[use]
print(f"{use.sum()} of {has_lat.sum()} points with an activation time are usable by every twin")
fits = {}
for name, (region, fixed) in twins.items():
    v, t0, cost = twin.calibrate_regions(V3, F3, region, pace3, pts, t_meas, v_init=0.8, fixed=fixed)
    fits[name] = (v, t0, np.sqrt(cost / len(pts)))
    print(f"{name:10s} speeds {np.round(v, 2)} m/s   offset {t0:5.1f} ms   RMSE {fits[name][2]:.2f} ms")

Read the speeds against the truth (1.0 normal, 0.6 slow, 0 scar). Every twin recovers the 25-ms offset and the
normal speed. The labeled twins get the slow speed wrong, and differently for each threshold: their low-voltage
region is not the true slow region (it misses part of it and includes the scar, which really blocks), and the fit
bends the slow speed to make up for that. A fitted speed is a property of the twin, not a measurement of the
tissue.

In [ ]:
# Checks
assert all(abs(t0 - OFFSET) < 1.5 for _, t0, _ in fits.values())          # offsets recovered
assert abs(fits["oracle"][0][0] - V_FAST) < 0.05 and abs(fits["oracle"][0][1] - V_SLOW) < 0.1
assert fits["1 region"][2] > max(fits["low < 0.5"][2], fits["low < 0.65"][2])  # one speed fits worse
assert fits["low < 0.5"][0][1] < V_SLOW - 0.05                                # the labeled twin's slow speed is biased
assert abs(fits["low < 0.5"][0][1] - fits["low < 0.65"][0][1]) > 0.03         # and depends on the threshold

## Part E · Is the twin any good?

Error on the fitted points flatters a twin. Use 5-fold cross-validation (eq-m33-holdout): fit without one fifth of
the points, predict them, repeat.

In [ ]:
def cv_error(region, fixed, seed=1):
    def fit(X, y):
        return twin.calibrate_regions(V3, F3, region, pace3, X, y, v_init=0.8, fixed=fixed)

    def predict(model, X):
        v, t0, _ = model
        return mp.eikonal(V3, F3, pace3, v[region])[X] + t0

    return twin.kfold_rmse(fit, predict, pts, t_meas, 5, np.random.default_rng(seed))


cv = {name: cv_error(*twins[name]) for name in twins}
for name in twins:
    print(f"{name:10s} fitted {fits[name][2]:.2f} ms   held-out {cv[name]:.2f} ms")

Now split the surface into many regions: 24 (12 slices of longitude, north and south), and fit them to only 60 of
the points, starting from three different random guesses. Look at how far the fitted speeds of each region move
from start to start, against how many points that region contains.

In [ ]:
sub = np.random.default_rng(7).choice(len(pts), 60, replace=False)
pts60, t60 = pts[sub], t_meas[sub]
lon3 = np.arctan2(c3[:, 1], c3[:, 0])
sectors = (((lon3 + np.pi) / (2 * np.pi) * 12).astype(int) % 12) + 12 * (c3[:, 2] > 0)
starts = []
for s in range(3):
    guess = np.random.default_rng(100 + s).uniform(0.4, 1.6, 24)
    v, t0, cost = twin.calibrate_regions(V3, F3, sectors, pace3, pts60, t60, v_init=guess, max_nfev=15)
    starts.append(v)
    print(f"start {s}: RMSE on its 60 points {np.sqrt(cost / 60):.2f} ms")
starts = np.array(starts)
spread = starts.max(axis=0) - starts.min(axis=0)
tri_of_point = np.array([np.flatnonzero((F3 == p).any(axis=1))[0] for p in pts60])
count = np.bincount(sectors[tri_of_point], minlength=24)
plt.figure(figsize=(6, 3))
plt.scatter(count, spread)
plt.xlabel("points inside the region"); plt.ylabel("spread of fitted speed (m/s)")
plt.title("24-region twin, three starts"); plt.tight_layout(); plt.show()

All three starts fit their points about equally well, yet regions with few points end with quite different
speeds. Those speeds are not determined by the data, so the twin has no business reporting them. A region with no
point inside can still be constrained a little, by points the wave reaches after crossing it.

In [ ]:
# Checks
assert cv["oracle"] <= min(cv["low < 0.5"], cv["low < 0.65"]) + 0.05          # the true structure predicts best
assert max(cv["low < 0.5"], cv["low < 0.65"]) < cv["1 region"]               # any voltage-based split beats one speed
assert all(cv[n] >= fits[n][2] - 0.05 for n in twins)                                 # held-out error is not smaller
assert spread[count <= 1].mean() > 3 * spread[count >= 4].mean()                      # sparse regions are not identified

## Part F · Hand over to openCARP

openCARP reads a mesh as three files with a shared base name (CARP/openCARP file-format documentation):

- `.pts`: a header line with the number of nodes, then one `x y z` line per node, in **micrometres**;
- `.elem`: a header line with the number of elements, then one line per element, `Tr n0 n1 n2 tag` for a
  triangle, with 0-based node indices and a region tag;
- `.lon`: a header line with the number of fibre vectors per element (1 here), then one vector per element.

This twin is isotropic, so the fibre direction does not matter; each triangle gets a direction in its own plane.

openCARP needs conductivities, not speeds. Conduction velocity scales with the square root of conductivity
(eq-m33-cv-scaling), so each region's conductivity is a reference conductivity times $(v_\text{region}/v_\text{ref})^2$.
The reference pair ($\sigma_\text{ref}$, $v_\text{ref}$) depends on the membrane model and mesh you choose in
openCARP, so you measure it there; the lab writes the scale factors.

In [ ]:
best = min(["low < 0.5", "low < 0.65"], key=cv.get)   # the labeled twin with the lower held-out error
region, fixed = twins[best]
v_fit = fits[best][0]
print("exporting the twin", repr(best))
out = Path(tempfile.mkdtemp()) / "m33_twin"
files = twin.write_opencarp(out, V3, F3, tags=region, fibres=twin.tangent_fibres(V3, F3))
scale = {int(r): (float(v_fit[r] / v_fit[0]) ** 2) for r in range(region.max() + 1)}
meta = {"twin": best, "tags": {"0": "normal voltage", "1": "low voltage"},
        "fitted_speed_m_per_s": {str(r): float(v_fit[r]) for r in range(region.max() + 1)},
        "conductivity_scale_vs_tag0": {str(k): v for k, v in scale.items()},
        "pacing_node": pace3,
        "note": "Scale the openCARP conductivities of each tag by these factors."}
out.with_suffix(".json").write_text(json.dumps(meta, indent=2))
for f in files + [out.with_suffix(".json")]:
    print(f.name, f.stat().st_size, "bytes")
print(open(files[0]).read().splitlines()[:2], open(files[1]).read().splitlines()[:2])

In [ ]:
# Checks
V_back, F_back, tags_back, fib_back = twin.read_opencarp(out)
assert np.allclose(V_back, V3, atol=1e-6) and np.array_equal(F_back, F3) and np.array_equal(tags_back, region)
assert int(open(files[0]).readline()) == len(V3)
assert abs(float(open(files[0]).read().splitlines()[1].split()[0]) - V3[0, 0] * 1000) < 1e-2   # micrometres
assert scale[0] == 1.0 and 0 < scale[1] < 1

## Running openCARP yourself

openCARP is not part of this course: its simulator and binaries are under the openCARP Academic Public License,
so the course links to it and never ships it. To continue on your own machine:

1. Install openCARP from <https://opencarp.org> and read its license.
2. Choose a membrane model; run a short strand in openCARP to measure $v_\text{ref}$ for a reference conductivity
   $\sigma_\text{ref}$ on a mesh like this one.
3. Give each tag its conductivity, $\sigma_\text{ref}$ times the scale factor in `m33_twin.json`, and pace at the
   node listed there. Parameter names are in the openCARP documentation
   (<https://opencarp.org/documentation/opencarp-parameters>).
4. Compare openCARP's activation times with the eikonal twin and with the measured points. Differences like those
   in demo m33-d2 are expected near the pacing site and around the scar.

Only your own simulation outputs may be shared under this course's licenses; openCARP itself may not.